<a href="https://colab.research.google.com/github/barr92/public-house-price-data/blob/main/UK%20Land%20Registry%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signals BI — Price Paid Aggregation Pipeline
**Project:** `signalsbi-new` | **Dataset:** `uk_sold_prices` (prod) / `uk_sold_prices_dev` (dev)

## How to use
| Scenario | Settings |
|---|---|
| First ever run | `ENV = 'dev'`, `FIRST_RUN = True` |
| Validate dev, promote to prod | Run Cell 6 with `PROMOTE = True` |
| Every month after | `ENV = 'prod'`, `FIRST_RUN = False` |

**You never need to manually download any files.**

## Tables to create (after run)
| Table | Description |
|---|---|
| `transactions_raw` | Source of truth — one row per sale, all history |
| `monthly_trends_sector` | Monthly volume + prices per sector |
| `monthly_trends_sector_by_type` | Monthly volume + prices per sector x property type |
| `new_dev_trends_sector` | New build vs existing monthly prices + MoM/YoY diffs |
| `sales_trends_sector` | Monthly sales counts per sector |
| `price_growth_sector` | 3m / 6m / 12m / 5yr / 10yr growth |
| `annual_price_growth_sector` | Annual prices all years, new build vs existing |
| `cagr_sector_by_type` | 3yr / 5yr / 10yr CAGR per sector x property type |
| `sector_percentile_bands` | P10-P90 + national / district rank |
| `price_growth_local_authority` | LA-level growth + top 20 ranks |
| `street_summary` | Street-level new build vs existing |

## 0. Install dependencies

In [1]:
# !pip install google-cloud-bigquery google-cloud-bigquery-storage pandas db-dtypes requests --quiet

import io
import time
import requests
import pandas as pd
from datetime import date, datetime, timezone
from dateutil.relativedelta import relativedelta
from google.cloud import bigquery

from google.colab import auth
auth.authenticate_user()

print('Imports OK')

Imports OK


## 1. Configuration
**This is the only cell you need to change between runs.**

In [2]:
# ── CHANGE THESE TWO SETTINGS EACH RUN ─────────────────────────────────────
ENV       = 'dev'   # 'dev' or 'prod'
FIRST_RUN = True    # True  = full history backfill 1995->present + create tables
                    # False = latest month only, merge into existing tables
# ────────────────────────────────────────────────────────────────────────────

PROJECT  = 'signalsbi-new'
DATASETS = {'prod': 'uk_sold_prices', 'dev': 'uk_sold_prices_dev'}
DATASET       = DATASETS[ENV]
D             = f'`{PROJECT}.{DATASET}`'
RAW_TABLE     = f'{PROJECT}.{DATASET}.transactions_raw'
STAGING_TABLE = f'{PROJECT}.{DATASET}.transactions_staging'

S3_BASE = 'https://price-paid-data.publicdata.landregistry.gov.uk'
MONTHLY_URL  = f'{S3_BASE}/pp-monthly-update-new-version.csv'
HISTORY_FROM = 1995

def yearly_url(yr): return f'{S3_BASE}/pp-{yr}.csv'

def get_latest_complete_month():
    today = date.today()
    return (today.replace(day=1) - relativedelta(months=1)
            if today.day >= 25
            else today.replace(day=1) - relativedelta(months=2))

latest_month = get_latest_complete_month()
date_to   = latest_month + relativedelta(months=1) - relativedelta(days=1)
date_from = latest_month - relativedelta(years=31)

print(f'Environment  : {ENV.upper()}')
print(f'First run    : {FIRST_RUN}')
print(f'Dataset      : {PROJECT}.{DATASET}')
print(f'Latest month : {latest_month.strftime("%B %Y")}')
print(f'Window       : {date_from} -> {date_to}')

Environment  : DEV
First run    : True
Dataset      : signalsbi-new.uk_sold_prices_dev
Latest month : April 2026
Window       : 1995-04-01 -> 2026-04-30


## 2. Create dataset and `transactions_raw` if they don't exist

In [12]:
client = bigquery.Client(project=PROJECT)

# Create dataset (no-op if already exists)
ds = bigquery.Dataset(f'{PROJECT}.{DATASET}')
ds.location = 'EU'
client.create_dataset(ds, exists_ok=True)
print(f'Dataset `{PROJECT}.{DATASET}` ready.')

# Create transactions_raw with schema, partitioning, clustering
client.query(f'''
CREATE TABLE IF NOT EXISTS `{RAW_TABLE}`
(
  transaction_id      STRING    NOT NULL,
  transaction_date    DATE,
  price               INT64,
  postcode            STRING,
  postcode_sector     STRING,
  postcode_district   STRING,
  property_type       STRING,
  tenure              STRING,
  new_build           BOOL,
  paon                STRING,
  saon                STRING,
  street              STRING,
  locality            STRING,
  town                STRING,
  local_authority     STRING,
  county              STRING,
  year_month          STRING,
  year                INT64,
  record_status        STRING,
  transaction_category STRING,
  ingested_at          TIMESTAMP
)
PARTITION BY DATE_TRUNC(transaction_date, MONTH)
CLUSTER BY postcode_sector, property_type
OPTIONS (require_partition_filter = false)
''').result()
print(f'Table `{RAW_TABLE}` ready.')

Dataset `signalsbi-new.uk_sold_prices_dev` ready.
Table `signalsbi-new.uk_sold_prices_dev.transactions_raw` ready.


## 3. Download & ingest HMLR CSV(s)
- **FIRST_RUN = True** : loops 1995 -> current year (yearly files) then pulls monthly update. One-time only.
- **FIRST_RUN = False** : downloads monthly update only (~20 MB). Run every month.

In [ ]:
COL_NAMES = [
    'transaction_id','price','transaction_date','postcode',
    'property_type','new_build','tenure',
    'paon','saon','street','locality','town',
    'local_authority','county','transaction_category','record_status'
]
PROP_MAP   = {'D':'Detached','S':'Semi-Detached','T':'Terraced','F':'Flat','O':'Other'}
TENURE_MAP = {'F':'Freehold','L':'Leasehold'}

MERGE_SQL = f'''
MERGE `{RAW_TABLE}` AS target
USING `{STAGING_TABLE}` AS source
  ON target.transaction_id = source.transaction_id
  AND DATE_TRUNC(target.transaction_date, MONTH) = DATE_TRUNC(source.transaction_date, MONTH)
WHEN MATCHED AND source.record_status = 'D' THEN DELETE
WHEN MATCHED AND source.record_status IN ('A','C') THEN UPDATE SET
  transaction_date=source.transaction_date,price=source.price,
  postcode=source.postcode,postcode_sector=source.postcode_sector,
  postcode_district=source.postcode_district,property_type=source.property_type,
  tenure=source.tenure,new_build=source.new_build,
  paon=source.paon,saon=source.saon,street=source.street,
  locality=source.locality,town=source.town,
  local_authority=source.local_authority,county=source.county,
  year_month=source.year_month,year=source.year,
  transaction_category=source.transaction_category,
  record_status=source.record_status,ingested_at=source.ingested_at
WHEN NOT MATCHED AND source.record_status != 'D' THEN INSERT (
  transaction_id,transaction_date,price,postcode,postcode_sector,postcode_district,
  property_type,tenure,new_build,paon,saon,street,locality,town,
  local_authority,county,year_month,year,transaction_category,record_status,ingested_at
) VALUES (
  source.transaction_id,source.transaction_date,source.price,
  source.postcode,source.postcode_sector,source.postcode_district,
  source.property_type,source.tenure,source.new_build,
  source.paon,source.saon,source.street,source.locality,source.town,
  source.local_authority,source.county,source.year_month,source.year,
  source.transaction_category,source.record_status,source.ingested_at
)
'''

def download_csv(url):
    print(f'  Downloading {url} ...', end=' ', flush=True)
    resp = requests.get(url, timeout=600, stream=True)
    resp.raise_for_status()
    chunks, mb = [], 0
    for chunk in resp.iter_content(1024*1024):
        chunks.append(chunk); mb += len(chunk)/1024/1024
    print(f'{mb:.0f} MB')
    return b''.join(chunks)

def parse_and_clean(raw_bytes):
    df = pd.read_csv(io.BytesIO(raw_bytes), header=None, names=COL_NAMES, dtype=str, encoding='utf-8')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce').dt.date
    df['price']            = pd.to_numeric(df['price'], errors='coerce').astype('Int64')
    df['new_build']        = df['new_build'].map({'Y':True,'N':False})
    df['property_type']    = df['property_type'].str.strip().map(PROP_MAP)
    df['tenure']           = df['tenure'].str.strip().map(TENURE_MAP)
    df['record_status']    = df['record_status'].str.strip()
    df['transaction_category'] = df['transaction_category'].str.strip()
    df['postcode']         = df['postcode'].str.strip().str.upper()
    df = df.dropna(subset=['transaction_id','transaction_date','price','postcode'])
    before = len(df)
    df = df.drop_duplicates(subset=['transaction_id'], keep='last')
    dupes = before - len(df)
    if dupes > 0:
        print(f'  Removed {dupes:,} duplicate transaction_ids')
    df['postcode_sector']   = df['postcode'].str.rsplit(' ',n=1).str[0] + ' ' + df['postcode'].str[-3]
    df['postcode_district'] = df['postcode'].str.split(' ').str[0]
    df['year_month']        = df['transaction_date'].apply(lambda d: d.strftime('%Y-%m') if d else None)
    df['year']              = df['transaction_date'].apply(lambda d: d.year if d else None)
    df['ingested_at']       = pd.Timestamp.now(tz='UTC')
    cols = ['transaction_id','transaction_date','price','postcode','postcode_sector',
            'postcode_district','property_type','tenure','new_build','paon','saon',
            'street','locality','town','local_authority','county',
            'year_month','year','transaction_category','record_status','ingested_at']
    return df[[c for c in cols if c in df.columns]]

def upsert(df, label):
    before = len(df)
    df = df.drop_duplicates(subset=['transaction_id'], keep='last')
    dupes = before - len(df)
    if dupes > 0:
        print(f'  Removed {dupes:,} duplicate transaction_ids')
    print(f'  Upserting {len(df):,} rows ({label})...')
    client.load_table_from_dataframe(
        df, STAGING_TABLE,
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE', autodetect=True)
    ).result()
    client.query(MERGE_SQL).result()
    print(f'  Merge complete ({label})')

# ── Main ingest ───────────────────────────────────────────────────────────
if FIRST_RUN:
    print('FIRST RUN — downloading full history 1995 -> present...')
    for yr in range(HISTORY_FROM, date.today().year):
        try:
            upsert(parse_and_clean(download_csv(yearly_url(yr))), str(yr))
            time.sleep(1)
        except Exception as e:
            print(f'  WARNING: {yr} failed — {e}. Skipping.')
    print('Downloading current-year monthly update...')
    upsert(parse_and_clean(download_csv(MONTHLY_URL)), 'monthly update')
else:
    print('MONTHLY UPDATE — downloading latest month only...')
    upsert(parse_and_clean(download_csv(MONTHLY_URL)), 'monthly update')

n = client.query(f'SELECT COUNT(*) AS n FROM `{RAW_TABLE}`').to_dataframe().iloc[0]['n']
print(f'\ntransactions_raw row count: {n:,}')

FIRST RUN — downloading full history 1995 -> present...
  Upserting 796,589 rows (1995)...
  Merge complete (1995)
  Upserting 964,802 rows (1996)...
  Merge complete (1996)
  Upserting 1,093,922 rows (1997)...
  Merge complete (1997)
  Upserting 1,050,000 rows (1998)...
  Merge complete (1998)
  Upserting 1,194,351 rows (1999)...
  Merge complete (1999)
  Upserting 1,128,730 rows (2000)...
  Merge complete (2000)
  Upserting 1,245,002 rows (2001)...
  Merge complete (2001)
  Upserting 1,350,794 rows (2002)...
  Merge complete (2002)
  Upserting 1,234,502 rows (2003)...
  Merge complete (2003)
  Upserting 1,231,256 rows (2004)...
  Merge complete (2004)
  Upserting 1,061,070 rows (2005)...
  Merge complete (2005)
  Upserting 1,325,474 rows (2006)...
  Merge complete (2006)
  Upserting 1,271,758 rows (2007)...
  Merge complete (2007)
  Upserting 649,199 rows (2008)...
  Merge complete (2008)
  Upserting 625,051 rows (2009)...
  Merge complete (2009)
  Upserting 663,030 rows (2010)...
  

## 6. Promote dev -> prod
Run this cell manually after validating dev output. Set `PROMOTE = True` to execute.

**Only run when `ENV = 'dev'` and you are happy with all row counts above.**

In [ ]:
### Skip for now - Run once we're done with cleaner

# PROMOTE = False  # <-- set True to copy dev -> prod

# if not PROMOTE:
#     print('Promotion skipped. Set PROMOTE = True to copy dev -> prod.')
# else:
#     if ENV != 'dev':
#         raise RuntimeError("Set ENV = 'dev' before promoting.")
#     prod_dataset = DATASETS['prod']
#     to_promote = [t for t, _ in TABLES] + ['transactions_staging']
#     for table in to_promote:
#         src = f'{PROJECT}.{DATASETS["dev"]}.{table}'
#         dst = f'{PROJECT}.{prod_dataset}.{table}'
#         print(f'  Copying {table}...', end=' ', flush=True)
#         try:
#             client.copy_table(src, dst, job_config=bigquery.CopyJobConfig(
#                 write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)).result()
#             print('done')
#         except Exception as e:
#             print(f'SKIPPED ({e})')
#     print(f'Promotion complete -> {PROJECT}.{prod_dataset}')